# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alsa-mirza/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding 1: Growing vs declining content:
This research paper shows that growing pages are comparitively longer and younger than declining pages. Growing pages average about 3.2K words and 184 days of age. On the other hand, declinig pages average about 2.3K words and 230 days. The paper reports this as a directional observational comparison.

Methodology question:
How specifically was the growth/decline label determined from the underlying search-performance data? I would want to see the criteria which determines what constitutes a page growing versus declining, in terms of the time window and threshold used. Furthermore, I would ask if the validation/comparison design was such that the same page or client history was not used on both sides of the comparison.
This is important to establish how much weight should be given to the observed associations with content length, age, and so on.


Finding 2: The content performance curve:
This research paper shows a content-age performance pattern in which content reaches its highest health score at 61–90 days, with a 33 score. The paper also shows a decline around 271–365 days, where the health score is 14 and is followed by a higher score of 25 for content aged 365+ days.

Methodology question:
How is the health score defined and calculated, and how are the content-age groups constructed? I would want to know whether the health score is calculated from observations after content falls within each age group, and whether the design of the experiment allows for multiple observations of the same pages or clients. I would also want clarification on whether the observed variation in age constitutes different variations of the pages rather than different types of pages entering into each age group.
This would provide a basis for my analysis of the curve and its implications for the effect of age on the performance of the pages. In particular, it would help identify whether the curve describes a directional relationship or merely an observed association.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

In week 5, I performed validation of the Random Forest model using the 80/20 training/test split. For this audit, I will use the client-grouped split to make sure that records from one client are not mixed in the test and training sets.

In this case, the validation design is a bit more conservative than the one used in week 5. This is because the models are tested against the data from clients that were not used for training explicitly. As such, I will see if the performance of the week-5 model holds true when the sample split is stricter.

In [16]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
print("HF_TOKEN loaded successfully:", HF_TOKEN is not None)

HF_TOKEN loaded successfully: True


In [17]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    split="train",
    token=HF_TOKEN
)
df = dataset.to_pandas()

print("Data loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Data loaded successfully!
Shape: (2414248, 21)

Columns:
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [18]:
# Check the columns needed to recreate the Week-5 target

print("Columns containing 'click':")
print([c for c in df.columns if "click" in c.lower()])

print("\nColumns containing 'impression':")
print([c for c in df.columns if "impression" in c.lower()])

print("\nColumns containing 'ctr':")
print([c for c in df.columns if "ctr" in c.lower()])

Columns containing 'click':
['clicks_90d', 'clicks_last30', 'clicks_prev30']

Columns containing 'impression':
['impressions_90d', 'impressions_last30', 'impressions_prev30', 'content_total_impressions_90d', 'rare_impressions_share', 'anonymized_impressions_share']

Columns containing 'ctr':
[]


In [19]:
# Recreate CTR and the Week-5 target from the available 90-day fields

df["ctr"] = df["clicks_90d"] / df["impressions_90d"].replace(0, pd.NA)
# Keep the same Week-5 threshold
df["target"] = (df["ctr"] < 0.05).astype(int)
print("CTR and target created successfully!")

print("\nTarget distribution:")
print(df["target"].value_counts())

print("\nTarget proportions:")
print(df["target"].value_counts(normalize=True))

CTR and target created successfully!

Target distribution:
target
1    2382540
0      31708
Name: count, dtype: int64

Target proportions:
target
1    0.986866
0    0.013134
Name: proportion, dtype: float64


In [20]:
# Week-5 feature set + client grouping column
features = [
    "impressions_90d",
    "clicks_90d",
    "impressions_last30",
    "clicks_last30",
    "query_char_count",
    "query_token_count"
]
group_column = "client_hash_id"
print("Checking Week-5 features...\n")

for col in features:
    print(f"{col}: {'FOUND' if col in df.columns else 'MISSING'}")
print(f"\n{group_column}: {'FOUND' if group_column in df.columns else 'MISSING'}")

print("\nFeature set:")
print(features)

print("\nNumber of unique clients:")
print(df[group_column].nunique())

Checking Week-5 features...

impressions_90d: FOUND
clicks_90d: FOUND
impressions_last30: FOUND
clicks_last30: FOUND
query_char_count: FOUND
query_token_count: FOUND

client_hash_id: FOUND

Feature set:
['impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'query_char_count', 'query_token_count']

Number of unique clients:
52


In [21]:
from sklearn.model_selection import GroupShuffleSplit

# Prepare the data for the honest split
audit_df = df[features + ["target", group_column]].dropna().copy()
X = audit_df[features]
y = audit_df["target"]
groups = audit_df[group_column]
# 80/20 split, but keeping each client entirely in one side
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)
train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
overlap = train_clients.intersection(test_clients)

print("=== HONEST CLIENT-GROUPED SPLIT ===")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(overlap))

if len(overlap) == 0:
    print("\nPASS: No client appears in both training and testing sets.")
else:
    print("\nWARNING: Client overlap detected.")

=== HONEST CLIENT-GROUPED SPLIT ===
Training rows: 2164186
Testing rows: 250062
Training clients: 41
Testing clients: 11
Client overlap: 0

PASS: No client appears in both training and testing sets.


In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Train the same type of model used in Week 5
honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
honest_model.fit(X_train, y_train)

# Predict on the unseen clients
honest_pred = honest_model.predict(X_test)

# Evaluate
honest_accuracy = accuracy_score(y_test, honest_pred)
honest_precision = precision_score(
    y_test,
    honest_pred,
    zero_division=0
)
honest_recall = recall_score(
    y_test,
    honest_pred,
    zero_division=0
)

print("=== WEEK-6 HONEST MODEL RESULTS ===")
print(f"Accuracy : {honest_accuracy:.4f}")
print(f"Precision: {honest_precision:.4f}")
print(f"Recall   : {honest_recall:.4f}")

=== WEEK-6 HONEST MODEL RESULTS ===
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000


In [23]:
comparison = pd.DataFrame({
    "Validation": [
        "Week-5 random 80/20 split",
        "Week-6 client-grouped split"
    ],
    "Accuracy": [
        0.9999,
        honest_accuracy
    ]
})
print("=== BEFORE / AFTER COMPARISON ===")
display(comparison)

=== BEFORE / AFTER COMPARISON ===


,Validation,Accuracy
0,Week-5 random 80/20 split,0.999900
1,Week-6 client-grouped split,0.999952


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [24]:
# Section 3: Leakage audit

target_column = "target"

features = [
    "impressions_90d",
    "clicks_90d",
    "impressions_last30",
    "clicks_last30",
    "query_char_count",
    "query_token_count"
]

print("Target column:", target_column)

print("\nFinal model features:")
for feature in features:
    print("-", feature)

print("\n1. Is the target column directly included as a feature?")
print(target_column in features)

print("\n2. How was the target created in Week 5?")
print('target = (ctr < 0.05).astype(int)')

print("\n3. Does the feature set contain the inputs used to calculate CTR?")
ctr_inputs = ["clicks_90d", "impressions_90d"]

for col in ctr_inputs:
    print(f"{col}: {col in features}")

print("\n4. Leakage assessment:")
if all(col in features for col in ctr_inputs):
    print(
        "REVIEW NEEDED: The target is derived from CTR, "
        "and the features include clicks_90d and impressions_90d, "
        "which are the inputs used to calculate CTR."
    )
else:
    print("No direct CTR-input leakage detected from the listed features.")

print("\nWeek-5 model accuracy was approximately 0.9999, "
      "so the near-perfect score should not be treated as evidence of strong generalization.")

Target column: target

Final model features:
- impressions_90d
- clicks_90d
- impressions_last30
- clicks_last30
- query_char_count
- query_token_count

1. Is the target column directly included as a feature?
False

2. How was the target created in Week 5?
target = (ctr < 0.05).astype(int)

3. Does the feature set contain the inputs used to calculate CTR?
clicks_90d: True
impressions_90d: True

4. Leakage assessment:
REVIEW NEEDED: The target is derived from CTR, and the features include clicks_90d and impressions_90d, which are the inputs used to calculate CTR.

Week-5 model accuracy was approximately 0.9999, so the near-perfect score should not be treated as evidence of strong generalization.


## 4. Claim rewrite

In Week-5 experiment, the Random Forest model showed extremely high accuracy on the tested split, although the target variable was formed using CTR while the features included clicks and impressions used to estimate it, introducing a significant leakage bias. It demonstrates an observed and potentially useful association, it should be regarded as a directional finding for decision support rather than an indication of generalizability or proof of concept about which pages should be optimized.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.